# Librerias

In [13]:
import json
import pandas as pd
from pathlib import Path

In [14]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

# Función genérica para cargar JSON

In [ ]:
def cargar_json(ruta) -> pd.DataFrame:
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data)

In [33]:

df_locales = cargar_json("../doc_files/json_files/locales202312.json")
df_licencias = cargar_json("../doc_files/json_files/licencias202312.json")
df_terrazas = cargar_json("../doc_files/json_files/terrazas202312.json")
df_actividadeconomica = cargar_json("../doc_files/json_files/actividadeconomica202312.json")


# Generacion de informe de exploracion

In [34]:
def explorar_dataset(nombre, df, output_md=True, ruta_salida=None):
    """
    Genera un reporte de exploración en Markdown ordenado y legible
    """

    md = []

    # Título
    md.append(f"# 📊 Exploración Dataset: {nombre}\n")

    # Información general
    md.append("## Información general\n")
    md.append(f"- **Registros:** {df.shape[0]:,}")
    md.append(f"- **Columnas:** {df.shape[1]}")
    md.append("")

    # Columnas
    md.append("## Columnas\n")
    columnas = pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str)
    })
    md.append(columnas.to_markdown(index=False))
    md.append("")

    # Nulos
    md.append("## Valores nulos\n")
    nulos = df.isnull().sum().reset_index()
    nulos.columns = ["columna", "nulos"]
    nulos = nulos.sort_values("nulos", ascending=False)
    md.append(nulos.to_markdown(index=False))
    md.append("")

    # Duplicados
    md.append("## Registros duplicados\n")
    duplicados = df.duplicated().sum()
    md.append(f"- **Duplicados:** {duplicados}")
    md.append("")

    # Head
    md.append("## Primeras filas (head)\n")
    md.append(df.head().to_markdown(index=False))
    md.append("")

    # Describe numérico
    md.append("## Estadísticas numéricas\n")
    md.append(df.describe().to_markdown())
    md.append("")

    # Describe categórico
    md.append("## Estadísticas categóricas\n")
    md.append(df.describe(include="object").to_markdown())
    md.append("")

    contenido = "\n".join(md)

    # Guardar archivo
    if output_md and ruta_salida:
        with open(ruta_salida, "w", encoding="utf-8") as f:
            f.write(contenido)

    return contenido

# ==========================
# LISTA DE DATASETS
# ==========================

datasets = {
    "locales": df_locales,
    "licencias": df_licencias,
    "terrazas": df_terrazas,
    "actividadeconomica": df_actividadeconomica,
}

for nombre, df in datasets.items():
    explorar_dataset(
        nombre.capitalize(),
        df,
        ruta_salida=f"reporte_{nombre}.md"
    )

C:\Users\braya\AppData\Local\Temp\ipykernel_32092\1402560101.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  md.append(df.describe(include="object").to_markdown())
C:\Users\braya\AppData\Local\Temp\ipykernel_32092\1402560101.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

# Busqueda de relaciones

3. Relaciones entre ficheros: identifica, siempre que sea posible, un campo o\
conjunto de campos que permita establecer relaciones entre los datos de los\
distintos ficheros. Considera que, al trabajar con datos no estructurados,\
estas relaciones podrían no ser evidentes o factibles.

- a. Si no es posible establecer una relación clara, analiza el contexto y\
propón una alternativa para manejar los datos, como almacenarlos de\
forma separada o diseñar estrategias para su integración futura.

In [35]:
columnas_comunes = (
    set(df_locales.columns)
    & set(df_licencias.columns)
    & set(df_terrazas.columns)
    & set(df_actividadeconomica.columns)
)

print(columnas_comunes)


{'coordenada_y_agrupacion', 'desc_distrito_local', 'fx_datos_ini', 'clase_vial_edificio', 'id_barrio_local', 'cal_edificio', 'desc_situacion_local', 'desc_vial_edificio', 'coordenada_x_local', 'id_distrito_local', 'id_ndp_edificio', 'coordenada_x_agrupacion', 'id_situacion_local', 'id_vial_edificio', 'fx_datos_fin', 'nom_edificio', 'id_clase_ndp_edificio', 'coordenada_y_local', 'id_local', 'rotulo', 'num_edificio', 'secuencial_local_PC', 'desc_tipo_acceso_local', 'id_local_agrupado', 'id_tipo_acceso_local', 'fx_carga', 'desc_barrio_local', 'id_planta_agrupado'}


In [36]:
df_locales["id_local"].isin(df_licencias["id_local"]).sum()


np.int64(74139)

In [37]:
df_locales["id_local"].isin(df_terrazas["id_local"]).sum()

np.int64(6767)

In [38]:
df_locales["id_local"].isin(df_actividadeconomica["id_local"]).sum()

np.int64(151161)

# Prueba de join

In [39]:
df_merge = df_locales.merge(df_licencias, on="id_local", how="inner")
print(df_merge.shape)

(150829, 96)


In [40]:
df_merge = df_locales.merge(df_terrazas, on="id_local", how="inner")
print(df_merge.shape)

(6767, 108)


In [41]:
df_merge = df_locales.merge(df_actividadeconomica, on="id_local", how="inner")
print(df_merge.shape)

(169559, 96)
